In [1]:
import pickle
import numpy as np
import os

file = open('inputs/wilson/random_weekeday_2.pkl', 'rb')
payload_wilson_initial = pickle.load(file)
file.close()

In [2]:
print(payload_wilson_initial)

{'requests': [{'booking_id': '1', 'pickup_pt': {'lon': -77.930793762, 'lat': 35.780387878}, 'dropoff_pt': {'lon': -77.893867493, 'lat': 35.719944}, 'pickup_time_window_start': 20043, 'pickup_time_window_end': 21843, 'dropoff_time_window_start': 20654.9, 'dropoff_time_window_end': 22454.9, 'am': 1, 'wc': 0}, {'booking_id': '2', 'pickup_pt': {'lon': -77.943908691, 'lat': 35.709342957}, 'dropoff_pt': {'lon': -77.996498108, 'lat': 35.733001709}, 'pickup_time_window_start': 20161, 'pickup_time_window_end': 21961, 'dropoff_time_window_start': 20684.5, 'dropoff_time_window_end': 22484.5, 'am': 1, 'wc': 0}, {'booking_id': '3', 'pickup_pt': {'lon': -77.913970947, 'lat': 35.719406128}, 'dropoff_pt': {'lon': -77.909812927, 'lat': 35.690387726}, 'pickup_time_window_start': 20212, 'pickup_time_window_end': 22012, 'dropoff_time_window_start': 20508.9, 'dropoff_time_window_end': 22308.9, 'am': 1, 'wc': 0}, {'booking_id': '4', 'pickup_pt': {'lon': -77.900054932, 'lat': 35.709503174}, 'dropoff_pt': {'l

In [3]:
from rtv_solver import OnlineRTVSolver

# Initialize the RTV solver with the URL of the OSRM server
online_rtv_solver = OnlineRTVSolver("http://127.0.0.1:5001/")

In [4]:
# creating a new payload with new requests
# consider all requests that start before 05:40:00

current_time = 5*3600+30*60
step = 10*60

selected_requests = []
for request in payload_wilson_initial["requests"]:
    if request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

3

In [5]:
# create a new payload with the selected requests
new_payload = {
    "depot": payload_wilson_initial["depot"],
    "requests": selected_requests,
    "driver_runs": payload_wilson_initial["driver_runs"],}

## Fast Heuristic method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_heuristic(new_payload)
unserved_requests

[]

In [6]:
# Simulate to 5:40:00

current_time += step
simulated_driver_runs = online_rtv_solver.simulate_manifest(current_time,new_driver_runs,intermediate_location=False)

In [7]:
# creating a new payload with new requests
# consider all requests that start between 05:40:00 and 05:50:00

selected_requests = []
for request in payload_wilson_initial["requests"]:
    if request["pickup_time_window_start"] >= current_time and request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

2

In [8]:
# create a new payload with the selected requests
new_payload = {
    "depot": payload_wilson_initial["depot"],
    "requests": selected_requests,
    "driver_runs": simulated_driver_runs,}

## Full RTV method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_rtv(new_payload)
unserved_requests

[]

In [9]:
# Simulate to 5:50:00

current_time += step
simulated_driver_runs = online_rtv_solver.simulate_manifest(current_time,new_driver_runs,intermediate_location=False)

In [10]:
# creating a new payload with new requests
# consider all requests that are between before 05:50:00 and after 05:60:00

selected_requests = []
for request in payload_wilson_initial["requests"]:
    if request["pickup_time_window_start"] >= current_time and request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

3

In [11]:
req = selected_requests[0]

# check feasibility of time slots


new_payload = {
    "depot": payload_wilson_initial["depot"], # JW: added to get it running
    "requests": [
    {
        'booking_id': req['booking_id'],
        'pickup_pt': req['pickup_pt'],
        'dropoff_pt': req['dropoff_pt'],
        'time_windows' : [
            {'pickup_time_window_start': req['pickup_time_window_start'], 'pickup_time_window_end': req['pickup_time_window_start'] + 60, 'dropoff_time_window_start': req['dropoff_time_window_start'], 'dropoff_time_window_end': req['dropoff_time_window_start'] + 180},
            {'pickup_time_window_start': req['pickup_time_window_start']+900, 'pickup_time_window_end': req['pickup_time_window_end']+900, 'dropoff_time_window_start': req['dropoff_time_window_start']+900, 'dropoff_time_window_end': req['dropoff_time_window_end']+900},
            {'pickup_time_window_start': req['pickup_time_window_start']+1800, 'pickup_time_window_end': req['pickup_time_window_end']+1800, 'dropoff_time_window_start': req['dropoff_time_window_start']+1800, 'dropoff_time_window_end': req['dropoff_time_window_end']+1800},
        ],
        'am': req['am'],
        'wc': req['wc']
    }],
    "driver_runs": simulated_driver_runs
}


feasible_windows = online_rtv_solver.check_feasibility(new_payload)
feasible_windows

{'depot': {'pt': {'lat': 35.723017652422435, 'lon': -77.90871990823223}, 'node_id': 0}, 'requests': [{'booking_id': '6', 'pickup_pt': {'lon': -77.964935303, 'lat': 35.75005722}, 'dropoff_pt': {'lon': -77.879508972, 'lat': 35.707454681}, 'time_windows': [{'pickup_time_window_start': 21066, 'pickup_time_window_end': 21126, 'dropoff_time_window_start': 21761, 'dropoff_time_window_end': 21941}, {'pickup_time_window_start': 21966, 'pickup_time_window_end': 23766, 'dropoff_time_window_start': 22661, 'dropoff_time_window_end': 24461}, {'pickup_time_window_start': 22866, 'pickup_time_window_end': 24666, 'dropoff_time_window_start': 23561, 'dropoff_time_window_end': 25361}], 'am': 1, 'wc': 0}], 'driver_runs': [{'state': {'run_id': 0, 'start_time': 18000, 'end_time': 72000, 'am_capacity': 8, 'wc_capacity': 3, 'locations_already_serviced': 2, 'location_dt_seconds': 20894.9, 'loc': {'lat': 35.719944, 'lon': -77.893867493, 'node_id': 6}, 'total_locations': 6}, 'manifest': [{'run_id': 0, 'booking_id

[({'pickup_time_window_start': 21966,
   'pickup_time_window_end': 23766,
   'dropoff_time_window_start': 22661,
   'dropoff_time_window_end': 24461},
  1.1359886201991465),
 ({'pickup_time_window_start': 22866,
   'pickup_time_window_end': 24666,
   'dropoff_time_window_start': 23561,
   'dropoff_time_window_end': 25361},
  1.1524893314367)]

In [12]:
feasible_windows[0] # this has the feasible time slot and associated VMT/PMT ratio for this slot

({'pickup_time_window_start': 21966,
  'pickup_time_window_end': 23766,
  'dropoff_time_window_start': 22661,
  'dropoff_time_window_end': 24461},
 1.1359886201991465)

In [ ]:
# Creating a request with infeasible time slots

req = selected_requests[0]

new_payload = {
    "depot": payload_wilson_initial["depot"],
    "requests": [
    {
        'booking_id': req['booking_id'],
        'pickup_pt': req['pickup_pt'],
        'dropoff_pt': req['dropoff_pt'],
        'pickup_time_window_start': req['pickup_time_window_start'], 
        'pickup_time_window_end': req['pickup_time_window_start']+180, 
        'dropoff_time_window_start': req['dropoff_time_window_start'], 
        'dropoff_time_window_end': req['dropoff_time_window_start']+180,
        'am': req['am'],
        'wc': req['wc']
    }],
    "driver_runs": simulated_driver_runs
}

## Full RTV method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_rtv(new_payload)
unserved_requests

['6']

In [14]:
# Serve at earliest possible time

# JW: API needs to fit the output, "new_unserved_requests" must be binded otherwise it breaks the next run
new_driver_runs, new_unserved_requests = online_rtv_solver.serve_asap(new_payload)

In [17]:
# Reoptimize the driver runs
# JW: optional: payload is incorrect

repotimized_driver_runs = online_rtv_solver.resolve_pdptw_rtv({"depot": payload_wilson_initial["depot"], "driver_runs": new_driver_runs, "requests": new_unserved_requests})

payload: {'depot': {'pt': {'lat': 35.723017652422435, 'lon': -77.90871990823223}, 'node_id': 0}, 'driver_runs': [{'state': {'run_id': 0, 'start_time': 18000, 'end_time': 72000, 'am_capacity': 8, 'wc_capacity': 3, 'locations_already_serviced': 2, 'location_dt_seconds': 20894.9, 'loc': {'lat': 35.719944, 'lon': -77.893867493, 'node_id': 6}, 'total_locations': 6}, 'manifest': [{'run_id': 0, 'booking_id': '1', 'order': 1, 'action': 'pickup', 'loc': {'lon': -77.930793762, 'lat': 35.780387878, 'node_id': 1}, 'scheduled_time': 20043, 'am': 1, 'wc': 0, 'time_window_start': 20043, 'time_window_end': 21843}, {'run_id': 0, 'booking_id': '1', 'order': 2, 'action': 'dropoff', 'loc': {'lat': 35.719944, 'lon': -77.893867493, 'node_id': 6}, 'scheduled_time': 20834.9, 'am': 1, 'wc': 0, 'time_window_start': 20654.9, 'time_window_end': 22454.9}, {'run_id': 0, 'booking_id': '4', 'order': 3, 'action': 'pickup', 'loc': {'lat': 35.709503174, 'lon': -77.900054932, 'node_id': 1}, 'scheduled_time': 21016.5, 'am